In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded!")

# Load dataset
df = pd.read_csv("RELEVAN_BERITA_SAHAM_FIX_BANGET.csv")

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

df['konten_clean'] = df['konten_clean'].fillna("").astype(str)
df = df[df['konten_clean'].str.split().str.len() >= 10].reset_index(drop=True)

# Simpan ID berita asli
df['doc_id'] = df.index

print(f"Total berita setelah filtering: {len(df)}")

Libraries loaded!
Total berita setelah filtering: 1510


In [2]:
MODEL_ID = "taufiqdp/indonesian-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)

device = 0 if torch.cuda.is_available() else -1

sentiment_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device
)

config.json:   0%|          | 0.00/922 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# NO CHUNK (FIX)

In [3]:
# Fungsi batch prediksi
def predict_batch(texts, batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        preds = sentiment_pipe(batch, truncation=True, max_length=512)
        results.extend(preds)
    return results

In [8]:
import re
# keyword saham
positif_keywords = [
    "menguat", "rebound", "bullish", "surplus", "naik", "melonjak",
    "penguatan", "stabil", "optimis", "pulih", "rekor", "melambung"
]
negatif_keywords = [
    "melemah", "anjlok", "bearish", "defisit", "turun", "merosot",
    "penurunan", "tertekan", "resesi", "krisis", "minus", "collapse", "pelemahan"
]

positif_pattern = re.compile(r"\b(" + "|".join(positif_keywords) + r")\b", re.IGNORECASE)
negatif_pattern = re.compile(r"\b(" + "|".join(negatif_keywords) + r")\b", re.IGNORECASE)

def apply_keyword_override(text, model_label, model_score):
    pos_match = bool(positif_pattern.search(text))
    neg_match = bool(negatif_pattern.search(text))

    if pos_match and not neg_match:
        return "positif", 1.0
    elif neg_match and not pos_match:
        return "negatif", 1.0
    elif pos_match and neg_match:
        # kalau kedua jenis keyword ada, pilih berdasarkan jumlah kemunculan
        pos_count = len(positif_pattern.findall(text))
        neg_count = len(negatif_pattern.findall(text))
        if pos_count > neg_count:
            return "positif", 1.0
        elif neg_count > pos_count:
            return "negatif", 1.0
        else:
            # seimbang -> fallback ke hasil model
            return model_label, model_score
    else:
        # tidak ada keyword -> pake hasil model
        return model_label, model_score

df["konten_clean"] = df["konten_clean"].fillna("").astype(str)
preds = predict_batch(df["konten_clean"].tolist(), batch_size=32)
df["sentiment_label_model"] = [p["label"] for p in preds]
df["sentiment_score_model"] = [float(p["score"]) for p in preds]

# ... (bagian mapping label model ke id) ...

# SEKARANG INI TIDAK AKAN ERROR KARENA FUNGSI SUDAH DI-DEFINE DI ATAS
adjusted = [
    apply_keyword_override(text, lbl_id, scr)
    for text, lbl_id, scr in zip(df["konten_clean"],
                                 df["sentiment_label_model_id"],
                                 df["sentiment_score_model"])
]
df["sentiment_label"], df["sentiment_score"] = zip(*adjusted)

In [9]:
df.to_csv("df_sentiment_saham_NO_CHUNK.csv", index=False)
print("Selesai — file ditulis: df_sentiment_saham_taufiqdp.csv")
print("Distribusi sentiment_label:", df["sentiment_label"].value_counts())

Selesai — file ditulis: df_sentiment_saham_taufiqdp.csv
Distribusi sentiment_label: sentiment_label
netral     636
positif    506
negatif    368
Name: count, dtype: int64


# CHUNK

In [10]:
import pandas as pd
import numpy as np
import torch
import random
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

# SET SEED (supaya stabil)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

df = pd.read_csv("RELEVAN_BERITA_SAHAM_FIX_BANGET.csv")

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

df['konten_clean'] = df['konten_clean'].fillna("").astype(str)
df = df[df['konten_clean'].str.split().str.len() >= 10].reset_index(drop=True)
df['doc_id'] = df.index

print(f"Total berita setelah filtering: {len(df)}")

MODEL_ID = "taufiqdp/indonesian-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)

model.eval()  # 🔥 WAJIB biar dropout mati

device = 0 if torch.cuda.is_available() else -1

sentiment_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device
)

# CHUNKING
MAX_LEN = 510

def split_chunks_token(text, tokenizer, max_len=510):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = [
        tokens[i:i+max_len]
        for i in range(0, len(tokens), max_len)
    ]
    return [tokenizer.decode(chunk) for chunk in chunks]

df['berita_chunks'] = df['konten_clean'].apply(
    lambda x: split_chunks_token(x, tokenizer, MAX_LEN)
)

df_chunks = df.explode('berita_chunks').reset_index(drop=True)

# Tambah nomor chunk biar jelas
df_chunks['chunk_id'] = df_chunks.groupby('doc_id').cumcount() + 1

print(f"Total chunk: {len(df_chunks)}")

# KEYWORD RULE
positive_keywords = [
    "menguat","rebound","bullish","surplus","naik","melonjak",
    "penguatan","stabil","optimis","pulih","rekor","melambung"
]

negative_keywords = [
    "melemah","anjlok","bearish","defisit","turun","merosot",
    "penurunan","tertekan","resesi","krisis","minus","pelemahan"
]

positive_keywords = [k.lower() for k in positive_keywords]
negative_keywords = [k.lower() for k in negative_keywords]

def keyword_sentiment(text):
    t = text.lower()
    for k in positive_keywords:
        if k in t:
            return "positif", 1.0
    for k in negative_keywords:
        if k in t:
            return "negatif", 1.0
    return None, None

keyword_results = df_chunks['berita_chunks'].apply(keyword_sentiment)
df_chunks['keyword_label'] = keyword_results.apply(lambda x: x[0])
df_chunks['keyword_score'] = keyword_results.apply(lambda x: x[1])

# MODEL PREDICTION
def normalize_label(label):
    l = label.lower()
    if 'neg' in l:
        return 'negatif'
    if 'pos' in l:
        return 'positif'
    return 'netral'

to_model = df_chunks['keyword_label'].isna()
texts_for_model = df_chunks.loc[to_model, 'berita_chunks'].tolist()

def predict_batch(texts, batch_size=32):
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Predicting"):
        batch = texts[i:i+batch_size]
        out = sentiment_pipe(
            batch,
            truncation=True,
            max_length=512,
            padding=True
        )
        preds.extend(out)
    return preds

print(f"Predicting {len(texts_for_model)} chunks...")
preds = predict_batch(texts_for_model)

mapped_labels = [normalize_label(p['label']) for p in preds]
mapped_scores = [float(p['score']) for p in preds]

df_chunks.loc[to_model, 'pred_label'] = mapped_labels
df_chunks.loc[to_model, 'pred_score'] = mapped_scores

# Yang dari keyword
df_chunks.loc[~to_model, 'pred_label'] = df_chunks.loc[~to_model, 'keyword_label']
df_chunks.loc[~to_model, 'pred_score'] = df_chunks.loc[~to_model, 'keyword_score']

Total berita setelah filtering: 1510


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total chunk: 1577
Predicting 903 chunks...


Predicting: 100%|██████████| 29/29 [00:18<00:00,  1.53it/s]


In [14]:
# def weighted_vote(group):
#     """Get label with highest weighted score"""
#     label_scores = group.groupby('pred_label')['pred_score'].sum()
#     max_score = label_scores.max()
#     top_labels = label_scores[label_scores == max_score].index.tolist()

#     if len(top_labels) > 1:
#         return "netral"
#     return top_labels[0]

# # Aggregate per document
# df_doc = df_chunks.groupby('doc_id').apply(
#     lambda g: pd.Series({
#         'sentiment_label': weighted_vote(g),
#         'sentiment_score': g['pred_score'].mean()
#     })
# ).reset_index()

def weighted_vote(group):
    label_scores = group.groupby('pred_label')['pred_score'].sum()
    max_score = label_scores.max()
    top_labels = label_scores[label_scores == max_score].index.tolist()

    if len(top_labels) > 1:
        return pd.Series({'label': 'netral', 'score': group['pred_score'].mean()})

    winning_label = top_labels[0]

    # Hanya chunk yang labelnya sama dengan label final
    winning_chunks = group[group['pred_label'] == winning_label]
    winning_score = winning_chunks['pred_score'].mean()

    return pd.Series({'label': winning_label, 'score': winning_score})

# Aggregate per document
df_doc = df_chunks.groupby('doc_id').apply(
    lambda g: weighted_vote(g).rename({'label': 'sentiment_label', 'score': 'sentiment_score'})
).reset_index()

# Merge ke df utama
df = df.merge(df_doc, on='doc_id')

# SAVE
df_chunks.to_csv("CHUNK_LEVEL_WITH_SCORE_WEIGHTED.csv", index=False)
df.to_csv("FINAL_SENTIMENT_KEYWORD_INDOBERT_WEIGHTED.csv", index=False)

print("\nSaved files!")


# STATISTICS
print("\n=== DISTRIBUSI SENTIMENT ===")
print(df['sentiment_label'].value_counts())

print("\n=== CONFIDENCE SCORE STATS ===")
print(df.groupby('sentiment_label')['sentiment_score'].describe())

print("\n=== CHECKSUM ===")
print(f"Total positif: {(df['sentiment_label'] == 'positif').sum()}")
print(f"Total negatif: {(df['sentiment_label'] == 'negatif').sum()}")
print(f"Total netral: {(df['sentiment_label'] == 'netral').sum()}")
print(f"Mean score: {df['sentiment_score'].mean():.6f}")

print("\nDone!")


Saved files!

=== DISTRIBUSI SENTIMENT ===
sentiment_label
positif    647
netral     600
negatif    263
Name: count, dtype: int64

=== CONFIDENCE SCORE STATS ===
                 count      mean       std       min       25%      50%  \
sentiment_label                                                           
negatif          263.0  0.935172  0.143383  0.394316  1.000000  1.00000   
netral           600.0  0.875652  0.141458  0.366392  0.818064  0.95096   
positif          647.0  0.927712  0.135913  0.384391  0.918537  1.00000   

                     75%  max  
sentiment_label                
negatif          1.00000  1.0  
netral           0.97522  1.0  
positif          1.00000  1.0  

=== CHECKSUM ===
Total positif: 647
Total negatif: 263
Total netral: 600
Mean score: 0.908325

Done!


# AGREGASI

## PAKE INI 3

In [15]:
# AGREGASI SENTIMEN HARIAN

def signed_sentiment_score(row):
    """
    S_i = signed confidence score
    positif: +confidence
    negatif: -confidence
    netral: 0
    """
    label = row['sentiment_label']
    conf = row['sentiment_score']

    if label == 'positif':
        return conf
    elif label == 'negatif':
        return -conf
    else:  # netral
        return 0.0

# Buat kolom S_i (sentiment individual)
df['S_i'] = df.apply(signed_sentiment_score, axis=1)

# Agregasi per tanggal
agg_df = df.groupby('tanggal').agg(
    avg_sentiment=('S_i', 'mean'),
    prop_pos=('sentiment_label', lambda x: (x=='positif').mean()),
    prop_neg=('sentiment_label', lambda x: (x=='negatif').mean()),
    prop_neu=('sentiment_label', lambda x: (x=='netral').mean()),
    count=('sentiment_label', 'count')  # n berita per hari
).reset_index()

# Max impact (label dominan)
def max_impact(row):
    """Return label dengan proporsi tertinggi"""
    props = {
        'positif': row['prop_pos'],
        'negatif': row['prop_neg'],
        'netral': row['prop_neu']
    }

    max_val = max(props.values())
    top_labels = [k for k, v in props.items() if v == max_val]

    # Tie-break: positif vs negatif → netral
    if len(top_labels) > 1 and 'positif' in top_labels and 'negatif' in top_labels:
        return 'netral'

    return top_labels[0]

agg_df['max_impact'] = agg_df.apply(max_impact, axis=1)

agg_df.to_csv("AGREGASI_SENTIMEN_HARIAN.csv", index=False)

print("\n=== AGREGASI HARIAN ===")
print(agg_df.head(10))

print("\n=== DISTRIBUSI MAX IMPACT ===")
print(agg_df['max_impact'].value_counts())

print("\n=== STATISTIK S_t ===")
print(agg_df['avg_sentiment'].describe())


=== AGREGASI HARIAN ===
      tanggal  avg_sentiment  prop_pos  prop_neg  prop_neu  count max_impact
0  2022-01-01       0.000000       0.0       0.0       1.0      1     netral
1  2022-01-02       0.000000       0.0       0.0       1.0      2     netral
2  2022-01-03       0.000000       0.0       0.0       1.0      1     netral
3  2022-01-04       0.789194       1.0       0.0       0.0      2    positif
4  2022-01-06      -0.646163       0.0       1.0       0.0      1    negatif
5  2022-01-07       0.500000       0.5       0.0       0.5      2    positif
6  2022-01-09       0.000000       0.0       0.0       1.0      1     netral
7  2022-01-11       1.000000       1.0       0.0       0.0      1    positif
8  2022-01-12       0.000000       0.0       0.0       1.0      1     netral
9  2022-01-13       0.000000       0.0       0.0       1.0      1     netral

=== DISTRIBUSI MAX IMPACT ===
max_impact
positif    368
netral     303
negatif    122
Name: count, dtype: int64

=== STATISTIK 

In [16]:
import pandas as pd
df_price = pd.read_csv("SAHAM_CLEAN_22-24.csv")
df_price.head()

,tanggal,Terakhir,Pembukaan,Tertinggi,Terendah,Vol.,Perubahan%
0,2022-01-03,6665.31,6586.26,6677.20,6586.13,1.826000e+10,1.27
1,2022-01-04,6695.37,6675.13,6720.66,6675.13,1.859000e+10,0.45
2,2022-01-05,6662.30,6703.17,6738.11,6634.84,1.791000e+10,-0.49
3,2022-01-06,6653.35,6674.85,6679.85,6593.23,1.800000e+10,-0.13
4,2022-01-07,6701.32,6669.51,6712.15,6647.71,1.571000e+10,0.72


In [17]:
df_final = df_price.merge(agg_df, on='tanggal', how='left')
df_final.head()

,tanggal,Terakhir,Pembukaan,Tertinggi,Terendah,Vol.,Perubahan%,avg_sentiment,prop_pos,prop_neg,prop_neu,count,max_impact
0,2022-01-03,6665.31,6586.26,6677.20,6586.13,1.826000e+10,1.27,0.000000,0.0,0.0,1.0,1.0,netral
1,2022-01-04,6695.37,6675.13,6720.66,6675.13,1.859000e+10,0.45,0.789194,1.0,0.0,0.0,2.0,positif
2,2022-01-05,6662.30,6703.17,6738.11,6634.84,1.791000e+10,-0.49,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-01-06,6653.35,6674.85,6679.85,6593.23,1.800000e+10,-0.13,-0.646163,0.0,1.0,0.0,1.0,negatif
4,2022-01-07,6701.32,6669.51,6712.15,6647.71,1.571000e+10,0.72,0.500000,0.5,0.0,0.5,2.0,positif


In [18]:
df_final['prop_pos'] = df_final['prop_pos'].fillna(0)
df_final['prop_neg'] = df_final['prop_neg'].fillna(0)
df_final['prop_neu'] = df_final['prop_neu'].fillna(1)
df_final['avg_sentiment'] = df_final['avg_sentiment'].fillna(0)
df_final['count'] = df_final['count'].fillna(0)
df_final['max_impact'] = df_final['max_impact'].fillna('netral')  # Default netral

In [19]:
df_final.head(3)

,tanggal,Terakhir,Pembukaan,Tertinggi,Terendah,Vol.,Perubahan%,avg_sentiment,prop_pos,prop_neg,prop_neu,count,max_impact
0,2022-01-03,6665.31,6586.26,6677.20,6586.13,1.826000e+10,1.27,0.000000,0.0,0.0,1.0,1.0,netral
1,2022-01-04,6695.37,6675.13,6720.66,6675.13,1.859000e+10,0.45,0.789194,1.0,0.0,0.0,2.0,positif
2,2022-01-05,6662.30,6703.17,6738.11,6634.84,1.791000e+10,-0.49,0.000000,0.0,0.0,1.0,0.0,netral


In [20]:
df_final.isna().sum()

,0
tanggal,1
Terakhir,1
Pembukaan,1
Tertinggi,1
Terendah,1
Vol.,3
Perubahan%,1
avg_sentiment,0
prop_pos,0
prop_neg,0


In [21]:
df_final.dropna(inplace=True)

In [22]:
df_final.to_csv("MERGE_SAHAM_GABUNGAN_FINAL.csv")

In [23]:
df['chunk_id'] = df.groupby('doc_id').cumcount() + 1

In [24]:
df[df['doc_id'] == 0][['doc_id','chunk_id','berita_chunks']]

,doc_id,chunk_id,berita_chunks
0,0,1,[jaksa agung burhanuddin puji menteri badan us...


# CONFIDENCE SCORE

## NO CHUNK

In [25]:
df = pd.read_csv("df_sentiment_saham_NO_CHUNK.csv")
df.head()

,tanggal,kategori,judul,url,konten,is_stock_related,konten_clean,doc_id,sentiment_label_model,sentiment_score_model,sentiment_label_model_id,sentiment_label,sentiment_score
0,2022-01-01,ekonomi,Jaksa Agung Puji Erick Thohir Bantu Bongkar Ko...,https://finance.detik.com/moneter/d-5880248/ja...,Jaksa Agung Burhanuddin memberikan pujian kepa...,True,jaksa agung burhanuddin puji menteri badan usa...,0,netral,0.909168,netral,netral,0.909168
1,2022-01-02,ekonomi,Aturan Harga hingga Penyaluran Premium Diromba...,https://finance.detik.com/energi/d-5881226/atu...,Presiden Joko Widodo (Jokowi) telah mengeluark...,True,presiden joko widodo jokowi keluar atur presid...,1,netral,0.965226,netral,netral,0.965226
2,2022-01-02,ekonomi,Presiden Joko Widodo (Jokowi) telah mengeluark...,https://finance.detik.com/energi/d-5881226/atu...,Presiden Joko Widodo (Jokowi) telah mengeluark...,True,presiden joko widodo jokowi keluar atur presid...,2,netral,0.965226,netral,netral,0.965226
3,2022-01-03,ekonomi,"Genjot Bisnis Data Center, Telkom Ambil Alih S...",https://finance.detik.com/bursa-dan-valas/d-58...,PT Telkom Indonesia (Persero) Tbk (TLKM) menga...,True,telkom indonesia persero tbk tlkm ambil alih s...,3,netral,0.962065,netral,netral,0.962065
4,2022-01-04,ekonomi,Ini Pesan Penting Luhut buat OJK,https://finance.detik.com/moneter/d-5884263/in...,Menteri Koordinator Kemaritiman dan Investasi ...,True,menteri koordinator maritim investasi luhut bi...,4,positif,0.789194,positif,positif,0.789194


In [26]:
df.groupby('sentiment_label')['sentiment_score'].agg(
    jumlah='count',
    mean_score='mean',
    std_score='std',
    min_score='min',
    max_score='max'
).round(3)

,jumlah,mean_score,std_score,min_score,max_score
sentiment_label,,,,,
negatif,368,0.947,0.131,0.394,1.000
netral,636,0.873,0.143,0.366,0.985
positif,506,0.893,0.155,0.384,1.000


In [27]:
(df['sentiment_score'] >= 0.7).mean() * 100

np.float64(85.89403973509934)

In [28]:
df.assign(high_conf=df['sentiment_score'] >= 0.7) \
  .groupby('sentiment_label')['high_conf'] \
  .mean() * 100

,high_conf
sentiment_label,
negatif,91.032609
netral,83.805031
positif,84.782609


## CHUNK

In [29]:
import pandas as pd
df=pd.read_csv("FINAL_SENTIMENT_KEYWORD_INDOBERT_WEIGHTED.csv")
df

,tanggal,kategori,judul,url,konten,is_stock_related,konten_clean,doc_id,berita_chunks,sentiment_label_x,sentiment_score_x,sentiment_label_y,sentiment_score_y,sentiment_label,sentiment_score
0,2022-01-01,ekonomi,Jaksa Agung Puji Erick Thohir Bantu Bongkar Ko...,https://finance.detik.com/moneter/d-5880248/ja...,Jaksa Agung Burhanuddin memberikan pujian kepa...,True,jaksa agung burhanuddin puji menteri badan usa...,0,['jaksa agung burhanuddin puji menteri badan u...,netral,0.909168,netral,0.909168,netral,0.909168
1,2022-01-02,ekonomi,Aturan Harga hingga Penyaluran Premium Diromba...,https://finance.detik.com/energi/d-5881226/atu...,Presiden Joko Widodo (Jokowi) telah mengeluark...,True,presiden joko widodo jokowi keluar atur presid...,1,['presiden joko widodo jokowi keluar atur pres...,netral,0.965226,netral,0.965226,netral,0.965226
2,2022-01-02,ekonomi,Presiden Joko Widodo (Jokowi) telah mengeluark...,https://finance.detik.com/energi/d-5881226/atu...,Presiden Joko Widodo (Jokowi) telah mengeluark...,True,presiden joko widodo jokowi keluar atur presid...,2,['presiden joko widodo jokowi keluar atur pres...,netral,0.965226,netral,0.965226,netral,0.965226
3,2022-01-03,ekonomi,"Genjot Bisnis Data Center, Telkom Ambil Alih S...",https://finance.detik.com/bursa-dan-valas/d-58...,PT Telkom Indonesia (Persero) Tbk (TLKM) menga...,True,telkom indonesia persero tbk tlkm ambil alih s...,3,['telkom indonesia persero tbk tlkm ambil alih...,netral,0.962065,netral,0.962065,netral,0.962065
4,2022-01-04,ekonomi,Ini Pesan Penting Luhut buat OJK,https://finance.detik.com/moneter/d-5884263/in...,Menteri Koordinator Kemaritiman dan Investasi ...,True,menteri koordinator maritim investasi luhut bi...,4,['menteri koordinator maritim investasi luhut ...,positif,0.789194,positif,0.789194,positif,0.789194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1505,2024-12-30,ekonomi,"Selain itu, KBIA juga diharapkan mampu menamba...",https://finance.detik.com/bursa-dan-valas/d-77...,"Selain itu, KBIA juga diharapkan mampu menamba...",True,selain kbia harap tambah eksposur investasi ne...,1505,['selain kbia harap tambah eksposur investasi ...,netral,0.886131,netral,0.886131,netral,0.886131
1506,2024-12-30,ekonomi,"Direktur Penilaian Perusahaan BEI, I Gede Nyom...",https://finance.detik.com/bursa-dan-valas/d-77...,"Direktur Penilaian Perusahaan BEI, I Gede Nyom...",True,direktur nilai usaha bei gede nyoman yetna seb...,1506,['direktur nilai usaha bei gede nyoman yetna s...,netral,0.960519,netral,0.960519,netral,0.960519
1507,2024-12-30,ekonomi,"Diketahui, sebelumnya terdapat 10 emiten yang ...",https://finance.detik.com/bursa-dan-valas/d-77...,"Diketahui, sebelumnya terdapat 10 emiten yang ...",True,tahu 10 emiten delisting mas murni indonesia t...,1507,['tahu 10 emiten delisting mas murni indonesia...,positif,1.000000,positif,1.000000,positif,1.000000
1508,2024-12-30,ekonomi,"Penjualan Naik, Emiten Gas Cetak Laba Rp 355 M",https://finance.detik.com/bursa-dan-valas/d-77...,Kinerja perusahaan ditopang pendapatan yang tu...,True,kerja usaha topang dapat tumbuh 37 9 yoy 189 6...,1508,['kerja usaha topang dapat tumbuh 37 9 yoy 189...,positif,1.000000,positif,1.000000,positif,1.000000


In [30]:
df.groupby('sentiment_label')['sentiment_score'].agg(
    jumlah='count',
    mean_score='mean',
    std_score='std',
    min_score='min',
    max_score='max'
).round(3)

,jumlah,mean_score,std_score,min_score,max_score
sentiment_label,,,,,
negatif,263,0.935,0.143,0.394,1.0
netral,600,0.876,0.141,0.366,1.0
positif,647,0.928,0.136,0.384,1.0


In [31]:
(df['sentiment_score'] >= 0.7).mean() * 100

np.float64(87.6158940397351)

In [32]:
df.assign(high_conf=df['sentiment_score'] >= 0.7) \
  .groupby('sentiment_label')['high_conf'] \
  .mean() * 100

,high_conf
sentiment_label,
negatif,88.973384
netral,84.333333
positif,90.108192
